<a href="https://colab.research.google.com/github/yalamanchilisravani08/Linear-algebra-tookit-/blob/main/Linear_algebra_tookit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q gradio sympy numpy

In [2]:
import gradio as gr
import sympy as sp
import numpy as np

# ============================================================
# LINEAR ALGEBRA TOOLKIT
# ============================================================

# ---------- Matrix Parser ----------
def parse_matrix(text):
    try:
        rows = text.strip().split("\n")
        matrix = []

        for row in rows:
            values = row.replace(",", " ").split()
            if not values:
                continue
            matrix.append([sp.Rational(v) for v in values])

        if not matrix:
            raise ValueError("Matrix is empty.")

        cols = len(matrix[0])

        if any(len(row) != cols for row in matrix):
            raise ValueError("All rows must have the same number of columns.")

        return sp.Matrix(matrix)

    except Exception as e:
        raise ValueError(
            "Invalid matrix. Enter numbers separated by spaces and rows on new lines."
        )


# ---------- Format Matrix ----------
def format_matrix(M):
    return sp.pretty(M)


# ---------- Rank ----------
def calculate_rank(matrix_text):
    try:
        A = parse_matrix(matrix_text)

        rank = A.rank()
        rref_matrix, pivots = A.rref()

        result = f"""
### Rank Result

**Input Matrix:**

{format_matrix(A)}

**Row Reduced Echelon Form:**

{format_matrix(rref_matrix)}

**Pivot Columns:** {pivots}

### Final Answer
**Rank = {rank}**

### Explanation
The rank of a matrix is the number of linearly independent rows or columns.
"""

        return result

    except Exception as e:
        return f"❌ Error: {str(e)}"


# ---------- Inverse ----------
def calculate_inverse(matrix_text):
    try:
        A = parse_matrix(matrix_text)

        if A.rows != A.cols:
            return "❌ Error: Inverse exists only for a square matrix."

        determinant = A.det()

        if determinant == 0:
            return f"""
### Inverse Result

{format_matrix(A)}

**Determinant = 0**

❌ This matrix is singular.

Therefore, **the inverse does not exist**.
"""

        inverse = A.inv()

        return f"""
### Inverse Result

**Input Matrix:**

{format_matrix(A)}

**Determinant:**

{determinant}

**Inverse Matrix:**

{format_matrix(inverse)}

### Final Answer
The inverse of the given matrix is shown above.

### Application
Matrix inverses are used for solving systems of linear equations,
computer graphics, engineering calculations and data transformations.
"""

    except Exception as e:
        return f"❌ Error: {str(e)}"


# ---------- Eigenvalues ----------
def calculate_eigen(matrix_text):
    try:
        A = parse_matrix(matrix_text)

        if A.rows != A.cols:
            return "❌ Error: Eigenvalues require a square matrix."

        eigenvalues = A.eigenvals()
        eigenvectors = A.eigenvects()

        result = f"""
### Eigenvalue Result

**Input Matrix:**

{format_matrix(A)}

### Eigenvalues

"""

        for value, multiplicity, vectors in eigenvectors:
            result += f"""
**λ = {value}**

Algebraic multiplicity: {multiplicity}

Eigenvector(s):

{format_matrix(sp.Matrix.hstack(*vectors))}
"""

        result += """
### Application

Eigenvalues and eigenvectors are used in:
- Artificial Intelligence
- Machine Learning
- Image processing
- Data analysis
- Recommendation systems
- Principal Component Analysis
"""

        return result

    except Exception as e:
        return f"❌ Error: {str(e)}"


# ---------- Diagonalization ----------
def calculate_diagonalization(matrix_text):
    try:
        A = parse_matrix(matrix_text)

        if A.rows != A.cols:
            return "❌ Error: Diagonalization requires a square matrix."

        try:
            P, D = A.diagonalize()

            return f"""
### Diagonalization Result

**Original Matrix A:**

{format_matrix(A)}

**Matrix P:**

{format_matrix(P)}

**Diagonal Matrix D:**

{format_matrix(D)}

### Relationship

**A = P D P⁻¹**

### Result

✅ The matrix is diagonalizable.

### Why is this useful?

Diagonalization makes many matrix calculations easier,
especially when calculating higher powers of matrices.
"""

        except Exception:
            return f"""
### Diagonalization Result

{format_matrix(A)}

❌ This matrix cannot be diagonalized using the available eigenvectors.

A matrix is diagonalizable when it has enough linearly independent eigenvectors.
"""

    except Exception as e:
        return f"❌ Error: {str(e)}"


# ---------- Cayley-Hamilton ----------
def calculate_cayley_hamilton(matrix_text):
    try:
        A = parse_matrix(matrix_text)

        if A.rows != A.cols:
            return "❌ Error: Cayley-Hamilton requires a square matrix."

        x = sp.symbols("x")

        char_poly = A.charpoly(x).as_expr()

        identity = sp.eye(A.rows)

        # Substitute the matrix into the characteristic polynomial
        poly = sp.Poly(char_poly, x)

        result_matrix = sp.zeros(A.rows)

        for (power,), coefficient in poly.terms():
            result_matrix += coefficient * (A ** power)

        return f"""
### Cayley-Hamilton Theorem

**Matrix A:**

{format_matrix(A)}

### Characteristic Polynomial

**p(λ) = {char_poly}**

### Substituting A into p(λ)

**p(A) =**

{format_matrix(result_matrix)}

### Verification

"""

        + (
            "✅ Cayley-Hamilton theorem verified: p(A) = 0."
            if result_matrix == sp.zeros(A.rows)
            else "⚠️ The result is not the zero matrix."
        )

    except Exception as e:
        return f"❌ Error: {str(e)}"


# ---------- Quadratic Form ----------
def calculate_quadratic(matrix_text):
    try:
        A = parse_matrix(matrix_text)

        if A.rows != A.cols:
            return "❌ Error: Quadratic form requires a square matrix."

        variables = sp.symbols("x1:" + str(A.rows + 1))
        X = sp.Matrix(variables)

        expression = (X.T * A * X)[0]

        return f"""
### Quadratic Form

**Matrix A:**

{format_matrix(A)}

### Quadratic Form

**Q(x) = xᵀ A x**

### Expanded Expression

**Q = {sp.expand(expression)}**

### Application

Quadratic forms are used in:
- Optimization
- Machine Learning
- Statistics
- Engineering
- Data analysis
"""

    except Exception as e:
        return f"❌ Error: {str(e)}"


# ---------- SVD ----------
def calculate_svd(matrix_text):
    try:
        A = parse_matrix(matrix_text)

        A_np = np.array(A.tolist(), dtype=float)

        U, singular_values, VT = np.linalg.svd(A_np)

        S = np.zeros_like(A_np, dtype=float)

        for i in range(len(singular_values)):
            S[i, i] = singular_values[i]

        return f"""
### Singular Value Decomposition

**Input Matrix A:**

{A}

### U Matrix

{np.array2string(U, precision=4, suppress_small=True)}

### Singular Values

{np.array2string(singular_values, precision=4, suppress_small=True)}

### Vᵀ Matrix

{np.array2string(VT, precision=4, suppress_small=True)}

### Formula

**A = U Σ Vᵀ**

### Application

SVD is used in:
- Image compression
- Noise reduction
- Recommendation systems
- Data analysis
- Dimensionality reduction
- Machine Learning
"""

    except Exception as e:
        return f"❌ Error: {str(e)}"


# ============================================================
# LEARNING SECTION
# ============================================================

learning_content = """
# 📚 Linear Algebra Learning Center

## 1. Rank

Rank tells us the number of linearly independent rows or columns
of a matrix.

**Used in:** data analysis, systems of equations and computer science.

---

## 2. Inverse

For a square matrix A, its inverse A⁻¹ satisfies:

**A × A⁻¹ = I**

The inverse exists only when:

**det(A) ≠ 0**

**Used in:** solving equations, graphics and engineering.

---

## 3. Eigenvalues & Eigenvectors

For a matrix A:

**Av = λv**

where λ is the eigenvalue and v is the eigenvector.

**Used in:** AI, machine learning, image processing and data science.

---

## 4. Diagonalization

A matrix can sometimes be represented as:

**A = P D P⁻¹**

where D is a diagonal matrix.

**Used in:** simplifying matrix computations.

---

## 5. Cayley-Hamilton Theorem

Every square matrix satisfies its own characteristic equation.

**Used in:** simplifying powers of matrices and mathematical models.

---

## 6. Quadratic Forms

A quadratic form can be written as:

**Q(x) = xᵀAx**

**Used in:** optimization, statistics and machine learning.

---

## 7. Singular Value Decomposition

A matrix can be decomposed as:

**A = UΣVᵀ**

**Used in:** image compression, recommendation systems,
noise reduction and dimensionality reduction.
"""


# ============================================================
# APPLICATION SECTION
# ============================================================

application_content = """
# 🌍 Real-World Applications

### 🤖 Artificial Intelligence
Linear algebra is one of the mathematical foundations of AI and Machine Learning.

### 🖼️ Image Processing
Matrices represent images. SVD can be used for image compression
and dimensionality reduction.

### 📊 Data Science
Matrices are used to represent and process large datasets.

### 🎯 Recommendation Systems
Eigenvalues, vectors and matrix techniques can help analyze relationships
between users and items.

### 🎮 Computer Graphics
Matrix transformations are used for rotation, scaling and movement
of objects.

### ⚙️ Engineering
Systems of linear equations and matrix operations are widely used
in engineering calculations.
"""


# ============================================================
# CUSTOM CSS
# ============================================================

css = """
body {
    background: #f5f7fb;
}

.gradio-container {
    max-width: 1200px !important;
}

.title {
    text-align: center;
}

.hero {
    text-align: center;
    padding: 25px;
    border-radius: 18px;
    background: linear-gradient(135deg, #4f46e5, #7c3aed);
    color: white;
    margin-bottom: 20px;
}

.card {
    padding: 20px;
    border-radius: 15px;
    background: white;
    border: 1px solid #e5e7eb;
}
"""


# ============================================================
# BUILD WEBSITE
# ============================================================

with gr.Blocks(
    title="Linear Algebra Toolkit",
    theme=gr.themes.Soft(),
    css=css
) as demo:

    # ---------- HOME ----------
    gr.HTML("""
    <div class="hero">
        <h1>🔢 Linear Algebra Toolkit</h1>
        <h3>Interactive Learning & Calculation Dashboard</h3>
        <p>
        Learn Linear Algebra concepts, calculate matrices step-by-step,
        and discover real-world applications.
        </p>
    </div>
    """)

    gr.Markdown("""
    ## 🚀 Welcome!

    This toolkit transforms Linear Algebra from a formula-based subject
    into an **interactive and practical learning experience**.

    ### Main Modules

    **Rank • Inverse • Eigenvalues • Diagonalization •
    Cayley-Hamilton • Quadratic Forms • SVD**
    """)

    # ---------- TABS ----------

    with gr.Tabs():

        # RANK
        with gr.Tab("📊 Rank"):

            gr.Markdown("## Rank Calculator")

            rank_input = gr.Textbox(
                label="Enter Matrix",
                value="1 2 3\n2 4 6\n1 1 1",
                lines=5
            )

            rank_button = gr.Button(
                "Calculate Rank",
                variant="primary"
            )

            rank_output = gr.Markdown()

            rank_button.click(
                calculate_rank,
                inputs=rank_input,
                outputs=rank_output
            )

        # INVERSE
        with gr.Tab("🔄 Inverse"):

            gr.Markdown("## Matrix Inverse Calculator")

            inverse_input = gr.Textbox(
                label="Enter Square Matrix",
                value="2 1\n1 3",
                lines=5
            )

            inverse_button = gr.Button(
                "Calculate Inverse",
                variant="primary"
            )

            inverse_output = gr.Markdown()

            inverse_button.click(
                calculate_inverse,
                inputs=inverse_input,
                outputs=inverse_output
            )

        # EIGENVALUES
        with gr.Tab("λ Eigenvalues"):

            gr.Markdown("## Eigenvalue & Eigenvector Calculator")

            eigen_input = gr.Textbox(
                label="Enter Square Matrix",
                value="4 1\n2 3",
                lines=5
            )

            eigen_button = gr.Button(
                "Calculate Eigenvalues",
                variant="primary"
            )

            eigen_output = gr.Markdown()

            eigen_button.click(
                calculate_eigen,
                inputs=eigen_input,
                outputs=eigen_output
            )

        # DIAGONALIZATION
        with gr.Tab("📐 Diagonalization"):

            gr.Markdown("## Matrix Diagonalization")

            diagonal_input = gr.Textbox(
                label="Enter Square Matrix",
                value="4 1\n2 3",
                lines=5
            )

            diagonal_button = gr.Button(
                "Diagonalize Matrix",
                variant="primary"
            )

            diagonal_output = gr.Markdown()

            diagonal_button.click(
                calculate_diagonalization,
                inputs=diagonal_input,
                outputs=diagonal_output
            )

        # CAYLEY HAMILTON
        with gr.Tab("🧮 Cayley-Hamilton"):

            gr.Markdown("## Cayley-Hamilton Theorem")

            ch_input = gr.Textbox(
                label="Enter Square Matrix",
                value="2 1\n1 2",
                lines=5
            )

            ch_button = gr.Button(
                "Verify Theorem",
                variant="primary"
            )

            ch_output = gr.Markdown()

            ch_button.click(
                calculate_cayley_hamilton,
                inputs=ch_input,
                outputs=ch_output
            )

        # QUADRATIC FORM
        with gr.Tab("📈 Quadratic Form"):

            gr.Markdown("## Quadratic Form Calculator")

            quadratic_input = gr.Textbox(
                label="Enter Square Matrix",
                value="2 1\n1 3",
                lines=5
            )

            quadratic_button = gr.Button(
                "Calculate Quadratic Form",
                variant="primary"
            )

            quadratic_output = gr.Markdown()

            quadratic_button.click(
                calculate_quadratic,
                inputs=quadratic_input,
                outputs=quadratic_output
            )

        # SVD
        with gr.Tab("🔬 SVD"):

            gr.Markdown("## Singular Value Decomposition")

            svd_input = gr.Textbox(
                label="Enter Matrix",
                value="1 2\n3 4",
                lines=5
            )

            svd_button = gr.Button(
                "Calculate SVD",
                variant="primary"
            )

            svd_output = gr.Markdown()

            svd_button.click(
                calculate_svd,
                inputs=svd_input,
                outputs=svd_output
            )

        # LEARNING
        with gr.Tab("📚 Learn"):

            gr.Markdown(learning_content)

        # APPLICATIONS
        with gr.Tab("🌍 Applications"):

            gr.Markdown(application_content)

    gr.Markdown("""
    ---
    ### 💡 Project Vision

    **Input → Calculation → Step-by-Step Explanation → Result → Real-World Application**

    This toolkit helps students understand not only **what the answer is**,
    but also **how it is obtained and where the concept is used**.

    **Developed as an educational project for Linear Algebra.**
    """)


# ============================================================
# LAUNCH
# ============================================================

demo.launch(share=True)

/tmp/ipykernel_1780/3792633704.py:534: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://5bb95e47b00ae782b5.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
